# Pipe end profile analyzer check

PCD/PLY 형태의 배관을 입력해서 직선 원통 구간들, 연결부 endpoint pair, 포지셔너에 고정할 양쪽 terminal end 프로파일을 확인하는 노트북입니다.

결과 구조:
- `segments`: 두 직선 원통 구간의 축, 반지름, endpoint
- `joint_endpoint_pairs`: 서로 연결된 것으로 판단된 endpoint pair
- `terminal_end_profiles`: 포지셔너 말단부에 고정할 양쪽 끝단 중심/축/원형 프로파일

In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import open3d as o3d

PLUGIN_ROOT = Path.cwd()
if PLUGIN_ROOT.name != "poseDeterminator":
    PLUGIN_ROOT = Path("python/plugins/poseDeterminator").resolve()

if str(PLUGIN_ROOT) not in sys.path:
    sys.path.insert(0, str(PLUGIN_ROOT))

from PipeEndProfileAnalyzer import analyze_pipe_end_profiles
from EndEffectorPoseOptimizer import EndEffectorPoseOptimizer

PLUGIN_ROOT

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


WindowsPath('d:/flame_robotics_drt/python/plugins/poseDeterminator')

## 1. 입력 파일 선택

`PIPE_PATH`를 확인하려는 PCD/PLY 파일로 바꾸면 됩니다. mm 단위 파일이면 `SCALE = 0.001`, 이미 m 단위이면 `SCALE = 1.0`으로 두세요.

In [2]:
PIPE_PATH = PLUGIN_ROOT / "data" / "PIPE NO.1.ply"
# PIPE_PATH = PLUGIN_ROOT / "data" / "PIPE NO.3_fill.ply"

SCALE = 1.0
VOXEL_SIZE = None       # 예: 0.002. 큰 파일이면 downsample 권장
MAX_POINTS = 25000
MAX_SEGMENTS = 6        # 직선 구간 최대 추정 개수
DISTANCE_THRESHOLD = None  # None이면 bbox 크기 기준 자동 추정

assert PIPE_PATH.exists(), PIPE_PATH
PIPE_PATH

WindowsPath('d:/flame_robotics_drt/python/plugins/poseDeterminator/data/PIPE NO.1.ply')

## 2. 양 끝단/프로파일 계산

In [3]:
result = analyze_pipe_end_profiles(
    PIPE_PATH,
    scale=SCALE,
    voxel_size=VOXEL_SIZE,
    max_points=MAX_POINTS,
    max_segments=MAX_SEGMENTS,
    ransac_iterations=250,
    sample_size=128,
    distance_threshold=DISTANCE_THRESHOLD,
    profile_sample_count=96,
    include_segment_points=True,
    random_seed=0,
)

summary = {
    "point_count": result["point_count"],
    "fit_point_count": result["fit_point_count"],
    "unassigned_fit_point_count": result["unassigned_fit_point_count"],
    "distance_threshold": result["distance_threshold"],
    "segments": [
        {
            "axis": seg["axis"],
            "radius": seg["radius"],
            "cylinder_fit_point_count": len(result.get("cylinder_fit_point_clouds", [[] for _ in result["segments"]])[idx]),
            "inlier_count": seg["inlier_count"],
            "rms_error": seg["rms_error"],
            "endpoints": seg["endpoints"],
        }
        for idx, seg in enumerate(result["segments"])
    ],
    "joint_endpoint_pairs": result["joint_endpoint_pairs"],
    "terminal_end_centers": [profile["center"] for profile in result["terminal_end_profiles"]],
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "point_count": 500000,
  "fit_point_count": 25000,
  "distance_threshold": 9.094846671585225,
  "segments": [
    {
      "axis": [
        -0.02497091621292991,
        0.9996697267354415,
        0.0060737790686363115
      ],
      "radius": 13.888875296591053,
      "inlier_count": 2687,
      "rms_error": 4.764262058707828,
      "endpoints": [
        [
          18.904182077571434,
          -948.530999548759,
          20.81034848011002
        ],
        [
          -4.648611394003547,
          -5.633493793881087,
          26.53919169818516
        ]
      ]
    },
    {
      "axis": [
        -0.027673217166364814,
        -0.9995439285316643,
        -0.01208834095938562
      ],
      "radius": 15.114981509745647,
      "inlier_count": 2577,
      "rms_error": 4.690602181919204,
      "endpoints": [
        [
          21.93135223595204,
          -6.3447520719757335,
          -15.733855946567754
        ],
        [
          -3.5825544460534218,
          -927.895

## 3. `EndEffectorPoseOptimizer` 래퍼로도 확인

실제 연동 코드에서는 아래처럼 optimizer에서 바로 호출할 수 있습니다.

In [4]:
optimizer = EndEffectorPoseOptimizer(debug_mode=True)
wrapped_result = optimizer.calculate_pipe_end_profiles(
    str(PIPE_PATH),
    scale=SCALE,
    voxel_size=VOXEL_SIZE,
    max_points=MAX_POINTS,
    max_segments=MAX_SEGMENTS,
    distance_threshold=DISTANCE_THRESHOLD,
    profile_sample_count=96,
    include_segment_points=True,
)

np.testing.assert_allclose(
    np.array([p["center"] for p in result["terminal_end_profiles"]]),
    np.array([p["center"] for p in wrapped_result["terminal_end_profiles"]]),
)
print("wrapper result matches direct analyzer")

wrapper result matches direct analyzer


## 4. Normal sampling only visualization

`calculate_pipe_profile()`의 초반 normal 기반 샘플링 단계만 따로 봅니다. 여기서는 전체 포인트, 기준점, normal vector, normal 추정 주변 포인트, normal 축 cylinder, 그 cylinder 내부 샘플링 포인트만 표시합니다.

In [ ]:
SELECTED_SEGMENT_INDEX_NORMAL_ONLY = 0
CONTEXT_MAX_POINTS = 8000


def _as_points(value):
    arr = np.asarray(value if value is not None else [], dtype=float)
    if arr.size == 0:
        return np.empty((0, 3), dtype=float)
    return arr.reshape((-1, 3))


def _as_vector(value):
    arr = np.asarray(value if value is not None else [], dtype=float).reshape(-1)
    return arr[:3] if arr.size >= 3 else np.empty((0,), dtype=float)


def _unit_vector(vector):
    vector = np.asarray(vector, dtype=float).reshape(3)
    norm = np.linalg.norm(vector)
    if norm <= 1e-12:
        raise ValueError(f"zero-length vector: {vector}")
    return vector / norm


def _rotation_matrix_from_vectors(source, target):
    source = _unit_vector(source)
    target = _unit_vector(target)
    cross = np.cross(source, target)
    dot = float(np.clip(np.dot(source, target), -1.0, 1.0))
    cross_norm = np.linalg.norm(cross)
    if cross_norm <= 1e-12:
        if dot > 0.0:
            return np.eye(3)
        axis = np.array([1.0, 0.0, 0.0])
        if abs(source[0]) > 0.9:
            axis = np.array([0.0, 1.0, 0.0])
        axis = _unit_vector(np.cross(source, axis))
        return o3d.geometry.get_rotation_matrix_from_axis_angle(axis * np.pi)
    axis = cross / cross_norm
    angle = np.arctan2(cross_norm, dot)
    return o3d.geometry.get_rotation_matrix_from_axis_angle(axis * angle)


def _make_point_cloud(points, color):
    pcd = o3d.geometry.PointCloud()
    points = _as_points(points)
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.paint_uniform_color(color)
    return pcd


def _make_sphere(center, radius, color):
    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=radius, resolution=24)
    sphere.translate(np.asarray(center, dtype=float).reshape(3))
    sphere.paint_uniform_color(color)
    return sphere


def _make_arrow(origin, direction, length, color):
    axis = _unit_vector(direction)
    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=max(length * 0.035, 1e-6),
        cone_radius=max(length * 0.075, 2e-6),
        cylinder_height=length * 0.75,
        cone_height=length * 0.25,
        resolution=32,
    )
    arrow.rotate(_rotation_matrix_from_vectors([0.0, 0.0, 1.0], axis), center=[0.0, 0.0, 0.0])
    arrow.translate(np.asarray(origin, dtype=float).reshape(3))
    arrow.paint_uniform_color(color)
    return arrow


def _make_sampling_cylinder(cylinder_debug, color):
    if not cylinder_debug:
        return None
    start = _as_vector(cylinder_debug.get("start"))
    axis = _as_vector(cylinder_debug.get("axis"))
    radius = float(cylinder_debug.get("radius", 0.0) or 0.0)
    height_range = cylinder_debug.get("height_range", [0.0, 0.0])
    if start.size != 3 or axis.size != 3 or radius <= 0.0 or len(height_range) != 2:
        return None
    axis_unit = _unit_vector(axis)
    height_min, height_max = float(height_range[0]), float(height_range[1])
    height = abs(height_max - height_min)
    if height <= 1e-12:
        return None
    center = start + axis_unit * ((height_min + height_max) * 0.5)
    mesh = o3d.geometry.TriangleMesh.create_cylinder(radius=radius, depth=height, resolution=64, split=8)
    mesh.rotate(_rotation_matrix_from_vectors([0.0, 0.0, 1.0], axis_unit), center=[0.0, 0.0, 0.0])
    mesh.translate(center)
    lines = o3d.geometry.LineSet.create_from_triangle_mesh(mesh)
    lines.paint_uniform_color(color)
    return lines


sampling_infos = result.get("sampling_debug_infos", []) if isinstance(result, dict) else []
if not sampling_infos:
    print("sampling_debug_infos가 없어서 PipeEndProfileAnalyzer를 reload한 뒤 include_segment_points=True로 다시 분석합니다.")
    import importlib
    import PipeEndProfileAnalyzer as pipe_analyzer_module

    pipe_analyzer_module = importlib.reload(pipe_analyzer_module)
    result = pipe_analyzer_module.analyze_pipe_end_profiles(
        PIPE_PATH,
        scale=SCALE,
        voxel_size=VOXEL_SIZE,
        max_points=MAX_POINTS,
        max_segments=MAX_SEGMENTS,
        ransac_iterations=250,
        sample_size=128,
        distance_threshold=DISTANCE_THRESHOLD,
        profile_sample_count=96,
        include_segment_points=True,
        random_seed=0,
    )
    sampling_infos = result.get("sampling_debug_infos", [])
if not sampling_infos:
    raise RuntimeError("sampling_debug_infos가 아직 없습니다. PipeEndProfileAnalyzer.py의 include_segment_points 경로를 확인하세요.")
if not (0 <= SELECTED_SEGMENT_INDEX_NORMAL_ONLY < len(sampling_infos)):
    raise IndexError(f"SELECTED_SEGMENT_INDEX_NORMAL_ONLY={SELECTED_SEGMENT_INDEX_NORMAL_ONLY}, available=0..{len(sampling_infos) - 1}")

sampling_debug = sampling_infos[SELECTED_SEGMENT_INDEX_NORMAL_ONLY]
sampling_cylinder_debug = sampling_debug.get("pipe_profile_sampling_cylinder") or {}

target_point = _as_vector(sampling_debug.get("pipe_profile_target_point"))
if target_point.size != 3:
    target_point = _as_vector(sampling_cylinder_debug.get("start"))

normal_axis = _as_vector(sampling_debug.get("pipe_profile_normal_axis"))
if normal_axis.size != 3:
    normal_axis = _as_vector(sampling_cylinder_debug.get("axis"))
if normal_axis.size != 3:
    normal_m = _as_vector(sampling_debug.get("normal_m"))
    normal_axis = -normal_m if normal_m.size == 3 else normal_axis

normal_estimation_points = _as_points(sampling_debug.get("normal_estimation_points"))
sampled_cylinder_points = _as_points(sampling_debug.get("pipe_profile_points_in_cylinder"))

if target_point.size != 3:
    raise RuntimeError("기준점 pipe_profile_target_point를 찾지 못했습니다.")
if normal_axis.size != 3:
    raise RuntimeError("normal axis를 찾지 못했습니다. sampling debug에 pipe_profile_normal_axis/axis가 비어 있습니다.")

context_pcd = o3d.io.read_point_cloud(str(PIPE_PATH))
if SCALE != 1.0:
    context_pcd.scale(SCALE, center=np.zeros(3))
if VOXEL_SIZE is not None and VOXEL_SIZE > 0:
    context_pcd = context_pcd.voxel_down_sample(VOXEL_SIZE)
if len(context_pcd.points) > CONTEXT_MAX_POINTS:
    context_pcd = context_pcd.random_down_sample(CONTEXT_MAX_POINTS / len(context_pcd.points))
context_pcd.paint_uniform_color([0.78, 0.78, 0.78])

extent = np.linalg.norm(context_pcd.get_axis_aligned_bounding_box().get_extent())
marker_radius = max(extent * 0.012, 1e-5)
arrow_length = max(extent * 0.12, marker_radius * 8.0)

geometries = [
    context_pcd,
    _make_point_cloud(normal_estimation_points, [1.0, 0.55, 0.0]),
    _make_point_cloud(sampled_cylinder_points, [0.05, 0.25, 1.0]),
    _make_sphere(target_point, marker_radius, [1.0, 0.0, 1.0]),
    _make_arrow(target_point, normal_axis, arrow_length, [1.0, 0.0, 1.0]),
]
sampling_cylinder = _make_sampling_cylinder(sampling_cylinder_debug, [0.0, 0.45, 1.0])
if sampling_cylinder is not None:
    geometries.append(sampling_cylinder)

print(f"selected segment: {SELECTED_SEGMENT_INDEX_NORMAL_ONLY}")
print(f"target point: {target_point}")
print(f"normal axis: {normal_axis}")
print(f"normal estimation points: {len(normal_estimation_points)}")
print(f"sampled points in normal-axis cylinder: {len(sampled_cylinder_points)}")
print(f"sampling cylinder: {sampling_cylinder_debug}")

o3d.visualization.draw_geometries(geometries)


## 4. Selected segment cylinder-fit input points

`SELECTED_SEGMENT_INDEX`를 바꾸면 해당 세그먼트의 `fit_cylinder(...)` 입력 포인트와 `calculate_pipe_profile()` 내부 샘플링 상태를 볼 수 있습니다. 옅은 회색은 전체 PCD, 주황색은 normal 추정 포인트/box, 빨간색은 실제 cylinder 최적화 입력점, 진한 회색은 refined inlier, 파란색은 샘플링용 얇은 cylinder/그 내부 점, 자홍색 화살표는 normal axis 방향, 초록색 wire sphere는 최종 sphere sampling 영역입니다.

In [ ]:
SELECTED_SEGMENT_INDEX = 0


def rotation_matrix_from_vectors(source, target):
    source = np.asarray(source, dtype=float)
    source = source / np.linalg.norm(source)
    target = np.asarray(target, dtype=float)
    target = target / np.linalg.norm(target)
    v = np.cross(source, target)
    c = np.dot(source, target)
    if np.linalg.norm(v) < 1e-12:
        return np.eye(3) if c > 0 else np.diag([1.0, -1.0, -1.0])
    vx = np.array([[0.0, -v[2], v[1]], [v[2], 0.0, -v[0]], [-v[1], v[0], 0.0]])
    return np.eye(3) + vx + vx @ vx * ((1.0 - c) / np.dot(v, v))


def make_line(points, color):
    points = np.asarray(points, dtype=float)
    line = o3d.geometry.LineSet()
    line.points = o3d.utility.Vector3dVector(points)
    line.lines = o3d.utility.Vector2iVector([[i, i + 1] for i in range(len(points) - 1)])
    line.colors = o3d.utility.Vector3dVector([color for _ in range(len(points) - 1)])
    return line


def make_sphere(center, radius, color):
    mesh = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
    mesh.translate(center)
    mesh.paint_uniform_color(color)
    return mesh


def make_cylinder_wireframe(segment, color):
    endpoints = np.array(segment["endpoints"], dtype=float)
    axis_vec = endpoints[1] - endpoints[0]
    height = np.linalg.norm(axis_vec)
    if height < 1e-12:
        return None
    cylinder = o3d.geometry.TriangleMesh.create_cylinder(
        radius=float(segment["radius"]), height=float(height), resolution=64, split=8
    )
    cylinder.rotate(rotation_matrix_from_vectors([0, 0, 1], axis_vec), center=(0, 0, 0))
    cylinder.translate((endpoints[0] + endpoints[1]) / 2.0)
    wire = o3d.geometry.LineSet.create_from_triangle_mesh(cylinder)
    wire.paint_uniform_color(color)
    return wire


def make_sampling_cylinder_wireframe(center, axis, radius, height, color):
    axis = np.asarray(axis, dtype=float)
    if np.linalg.norm(axis) < 1e-12:
        return None
    cylinder = o3d.geometry.TriangleMesh.create_cylinder(
        radius=float(radius), height=float(height), resolution=32, split=4
    )
    cylinder.rotate(rotation_matrix_from_vectors([0, 0, 1], axis), center=(0, 0, 0))
    cylinder.translate(center)
    wire = o3d.geometry.LineSet.create_from_triangle_mesh(cylinder)
    wire.paint_uniform_color(color)
    return wire


def make_sampling_cylinder_from_debug(cylinder_debug, color):
    if not cylinder_debug:
        return None
    start = np.array(cylinder_debug["start"], dtype=float)
    axis = np.array(cylinder_debug["axis"], dtype=float)
    height_min, height_max = cylinder_debug["height_range"]
    height = float(height_max - height_min)
    center = start + axis / np.linalg.norm(axis) * ((height_min + height_max) * 0.5)
    return make_sampling_cylinder_wireframe(center, axis, cylinder_debug["radius"], height, color)


def make_arrow(origin, direction, length, color):
    direction = np.asarray(direction, dtype=float)
    norm = np.linalg.norm(direction)
    if norm < 1e-12:
        return None
    direction = direction / norm
    radius = max(length * 0.025, 1e-5)
    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=radius,
        cone_radius=radius * 2.5,
        cylinder_height=length * 0.78,
        cone_height=length * 0.22,
        resolution=24,
    )
    arrow.rotate(rotation_matrix_from_vectors([0, 0, 1], direction), center=(0, 0, 0))
    arrow.translate(origin)
    arrow.paint_uniform_color(color)
    return arrow


def make_aabb_wireframe(min_bound, max_bound, color):
    box = o3d.geometry.AxisAlignedBoundingBox(min_bound, max_bound)
    line_set = o3d.geometry.LineSet.create_from_axis_aligned_bounding_box(box)
    line_set.paint_uniform_color(color)
    return line_set


assert "cylinder_fit_point_clouds" in result, "Run analysis with include_segment_points=True first."
assert 0 <= SELECTED_SEGMENT_INDEX < len(result["segments"])

full_context_pcd = o3d.io.read_point_cloud(str(PIPE_PATH))
if SCALE != 1.0:
    full_context_pcd.scale(SCALE, np.zeros(3))
if VOXEL_SIZE is not None and VOXEL_SIZE > 0:
    full_context_pcd = full_context_pcd.voxel_down_sample(VOXEL_SIZE)
CONTEXT_MAX_POINTS = 8000
if len(full_context_pcd.points) > CONTEXT_MAX_POINTS:
    full_context_pcd = full_context_pcd.random_down_sample(CONTEXT_MAX_POINTS / len(full_context_pcd.points))
full_context_pcd.paint_uniform_color([0.82, 0.82, 0.82])

segment = result["segments"][SELECTED_SEGMENT_INDEX]
fit_points_for_segment = np.array(result["cylinder_fit_point_clouds"][SELECTED_SEGMENT_INDEX], dtype=float)
refined_points_for_segment = np.array(result["segment_point_clouds"][SELECTED_SEGMENT_INDEX], dtype=float)
sampling_debug = result.get("sampling_debug_infos", [{}])[SELECTED_SEGMENT_INDEX]

fit_pcd = o3d.geometry.PointCloud()
fit_pcd.points = o3d.utility.Vector3dVector(fit_points_for_segment)
fit_pcd.paint_uniform_color([1.0, 0.1, 0.05])

refined_pcd = o3d.geometry.PointCloud()
refined_pcd.points = o3d.utility.Vector3dVector(refined_points_for_segment)
refined_pcd.paint_uniform_color([0.2, 0.2, 0.2])

cylinder_wire = make_cylinder_wireframe(segment, [0.0, 0.9, 1.0])
axis_line = make_line(np.array(segment["endpoints"], dtype=float), [1.0, 1.0, 0.0])

sampling_geoms = []
sampling_cylinder_debug = sampling_debug.get("pipe_profile_sampling_cylinder")
target_point = np.array(sampling_debug.get("pipe_profile_target_point", []), dtype=float)
normal_axis = np.array(sampling_debug.get("pipe_profile_normal_axis", []), dtype=float)
if normal_axis.shape != (3,) and sampling_cylinder_debug:
    normal_axis = np.array(sampling_cylinder_debug.get("axis", []), dtype=float)
if target_point.shape != (3,) and sampling_cylinder_debug:
    target_point = np.array(sampling_cylinder_debug.get("start", []), dtype=float)
points_in_cylinder = np.array(sampling_debug.get("pipe_profile_points_in_cylinder", []), dtype=float)
normal_points = np.array(sampling_debug.get("normal_estimation_points", []), dtype=float)
normal_sampling_box = sampling_debug.get("normal_estimation_sampling_box")
estimated_center = np.array(sampling_debug.get("estimated_center", []), dtype=float)
estimated_radius = float(sampling_debug.get("estimated_radius", 0.0) or 0.0)
if len(normal_points) > 0:
    normal_points_pcd = o3d.geometry.PointCloud()
    normal_points_pcd.points = o3d.utility.Vector3dVector(normal_points)
    normal_points_pcd.paint_uniform_color([1.0, 0.55, 0.0])
    sampling_geoms.append(normal_points_pcd)
if normal_sampling_box is not None:
    sampling_geoms.append(make_aabb_wireframe(
        np.array(normal_sampling_box[0], dtype=float),
        np.array(normal_sampling_box[1], dtype=float),
        [1.0, 0.55, 0.0],
    ))
if len(points_in_cylinder) > 0:
    cyl_points_pcd = o3d.geometry.PointCloud()
    cyl_points_pcd.points = o3d.utility.Vector3dVector(points_in_cylinder)
    cyl_points_pcd.paint_uniform_color([0.05, 0.25, 1.0])
    sampling_geoms.append(cyl_points_pcd)
if target_point.shape == (3,) and normal_axis.shape == (3,):
    normal_len = max(estimated_radius * 3.0, np.linalg.norm(np.asarray(segment["endpoints"][1]) - np.asarray(segment["endpoints"][0])) * 0.1, 1e-4)
    sampling_geoms.append(make_sphere(target_point, max(normal_len * 0.06, 1e-4), [1.0, 0.0, 1.0]))
    sampling_geoms.append(make_line([target_point, target_point + normal_axis / np.linalg.norm(normal_axis) * normal_len], [1.0, 0.0, 1.0]))
    normal_arrow = make_arrow(target_point, normal_axis, normal_len, [1.0, 0.0, 1.0])
    if normal_arrow is not None:
        sampling_geoms.append(normal_arrow)
sampling_cyl = make_sampling_cylinder_from_debug(sampling_cylinder_debug, [0.05, 0.25, 1.0])
if sampling_cyl is not None:
    sampling_geoms.append(sampling_cyl)
if estimated_center.shape == (3,) and estimated_radius > 0:
    sampling_sphere = o3d.geometry.TriangleMesh.create_sphere(radius=estimated_radius)
    sampling_sphere.translate(estimated_center)
    sampling_sphere_wire = o3d.geometry.LineSet.create_from_triangle_mesh(sampling_sphere)
    sampling_sphere_wire.paint_uniform_color([0.0, 0.8, 0.2])
    sampling_geoms.append(sampling_sphere_wire)

print(f"selected segment: {SELECTED_SEGMENT_INDEX}")
print(f"points passed to fit_cylinder: {len(fit_points_for_segment)}")
print(f"refined inlier points after fit: {len(refined_points_for_segment)}")
print(f"radius: {segment['radius']}")
print(f"axis: {np.array(segment['axis'])}")
print(f"rms_error: {segment['rms_error']}")
print(f"sampling normal axis: {normal_axis}")
print(f"sampling debug keys: {sorted(sampling_debug.keys())}")
print(f"points used to estimate normal: {len(normal_points)}")
print(f"points in sampling cylinder: {len(points_in_cylinder)}")
print(f"sampling cylinder debug: {sampling_cylinder_debug}")
print(f"estimated center/radius: {estimated_center}, {estimated_radius}")

geoms = [full_context_pcd, refined_pcd, fit_pcd, axis_line] + sampling_geoms
if cylinder_wire is not None:
    geoms.append(cylinder_wire)
o3d.visualization.draw_geometries(geoms)

## 5. Segment-only visualization

This view shows only fitted segment inlier points, fitted cylinder wireframes, each segment axis, endpoint markers, and unassigned fit points. Endpoint labels mean `seg{segment_index}/end{endpoint_index}`. Use this first when tuning `MAX_SEGMENTS`, `DISTANCE_THRESHOLD`, or `MIN_SEGMENT_POINTS`.

In [ ]:
def make_sphere(center, radius, color):
    mesh = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
    mesh.translate(center)
    mesh.paint_uniform_color(color)
    return mesh


def make_line(points, color):
    points = np.asarray(points, dtype=float)
    line = o3d.geometry.LineSet()
    line.points = o3d.utility.Vector3dVector(points)
    line.lines = o3d.utility.Vector2iVector([[i, i + 1] for i in range(len(points) - 1)])
    line.colors = o3d.utility.Vector3dVector([color for _ in range(len(points) - 1)])
    return line


def rotation_matrix_from_vectors(source, target):
    source = np.asarray(source, dtype=float)
    source = source / np.linalg.norm(source)
    target = np.asarray(target, dtype=float)
    target = target / np.linalg.norm(target)
    v = np.cross(source, target)
    c = np.dot(source, target)
    if np.linalg.norm(v) < 1e-12:
        return np.eye(3) if c > 0 else np.diag([1.0, -1.0, -1.0])
    vx = np.array([
        [0.0, -v[2], v[1]],
        [v[2], 0.0, -v[0]],
        [-v[1], v[0], 0.0],
    ])
    return np.eye(3) + vx + vx @ vx * ((1.0 - c) / np.dot(v, v))


def make_cylinder_wireframe(segment, color):
    endpoints = np.array(segment["endpoints"], dtype=float)
    axis_vec = endpoints[1] - endpoints[0]
    height = np.linalg.norm(axis_vec)
    if height < 1e-12:
        return None
    cylinder = o3d.geometry.TriangleMesh.create_cylinder(
        radius=float(segment["radius"]),
        height=float(height),
        resolution=48,
        split=8,
    )
    cylinder.rotate(rotation_matrix_from_vectors([0, 0, 1], axis_vec), center=(0, 0, 0))
    cylinder.translate((endpoints[0] + endpoints[1]) / 2.0)
    wire = o3d.geometry.LineSet.create_from_triangle_mesh(cylinder)
    wire.paint_uniform_color(color)
    return wire


pcd_for_scale = o3d.io.read_point_cloud(str(PIPE_PATH))
if SCALE != 1.0:
    pcd_for_scale.scale(SCALE, np.zeros(3))
if VOXEL_SIZE is not None and VOXEL_SIZE > 0:
    pcd_for_scale = pcd_for_scale.voxel_down_sample(VOXEL_SIZE)
bbox = pcd_for_scale.get_axis_aligned_bounding_box()
marker_radius = max(np.linalg.norm(bbox.get_extent()) * 0.01, 1e-4)

segment_colors = [
    [1.0, 0.15, 0.12],
    [0.1, 0.35, 1.0],
    [0.1, 0.75, 0.25],
    [1.0, 0.55, 0.05],
    [0.55, 0.2, 0.95],
    [0.0, 0.75, 0.8],
]

def make_label(text, position, color=(1.0, 1.0, 1.0)):
    try:
        label = o3d.t.geometry.TriangleMesh.create_text(text, depth=0.001).to_legacy()
        label.paint_uniform_color(color)
        label.scale(marker_radius * 8.0, center=(0, 0, 0))
        label.translate(position)
        return label
    except Exception:
        return None


segment_only_geometries = []

for idx, segment_points in enumerate(result.get("segment_point_clouds", [])):
    points = np.array(segment_points, dtype=float)
    segment_pcd = o3d.geometry.PointCloud()
    segment_pcd.points = o3d.utility.Vector3dVector(points)
    color = segment_colors[idx % len(segment_colors)]
    segment_pcd.paint_uniform_color(color)
    segment_only_geometries.append(segment_pcd)

    segment = result["segments"][idx]
    fitted_cylinder = make_cylinder_wireframe(segment, color)
    if fitted_cylinder is not None:
        segment_only_geometries.append(fitted_cylinder)

    endpoints = np.array(segment["endpoints"], dtype=float)
    segment_only_geometries.append(make_line(endpoints, color))
    for endpoint_idx, endpoint in enumerate(endpoints):
        segment_only_geometries.append(make_sphere(endpoint, marker_radius * 0.8, color))
        label = make_label(f"seg{idx}/end{endpoint_idx}", endpoint + np.array([marker_radius, marker_radius, marker_radius]), color)
        if label is not None:
            segment_only_geometries.append(label)

unassigned_points = np.array(result.get("unassigned_fit_points", []), dtype=float)
if len(unassigned_points) > 0:
    unassigned_pcd = o3d.geometry.PointCloud()
    unassigned_pcd.points = o3d.utility.Vector3dVector(unassigned_points)
    unassigned_pcd.paint_uniform_color([0.05, 0.05, 0.05])
    segment_only_geometries.append(unassigned_pcd)

print(f"segments: {len(result['segments'])}")
for idx, seg in enumerate(result["segments"]):
    print(
        f"S{idx}: inliers={seg['inlier_count']}, "
        f"radius={seg['radius']:.6g}, rms={seg['rms_error']:.6g}, "
        f"axis={np.array(seg['axis'])}"
    )
print(f"unassigned fit points: {result['unassigned_fit_point_count']}")

o3d.visualization.draw_geometries(segment_only_geometries)

## 4. Open3D 시각화

색상 의미:
- 회색: 입력 포인트클라우드
- 색상 포인트: 세그먼트별 inlier point
- 같은 색 선: 해당 세그먼트 축 endpoint 연결선
- 검정 포인트: 어느 세그먼트에도 할당되지 않은 fit point
- 노랑 구: 자유 끝단 중심
- 초록 원: 자유 끝단 원형 프로파일
- 자홍 구: 연결부 쪽 endpoint

In [ ]:
def make_sphere(center, radius, color):
    mesh = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
    mesh.translate(center)
    mesh.paint_uniform_color(color)
    return mesh


def make_line(points, color):
    points = np.asarray(points, dtype=float)
    line = o3d.geometry.LineSet()
    line.points = o3d.utility.Vector3dVector(points)
    line.lines = o3d.utility.Vector2iVector([[i, i + 1] for i in range(len(points) - 1)])
    line.colors = o3d.utility.Vector3dVector([color for _ in range(len(points) - 1)])
    return line


pcd = o3d.io.read_point_cloud(str(PIPE_PATH))
if SCALE != 1.0:
    pcd.scale(SCALE, np.zeros(3))
if VOXEL_SIZE is not None and VOXEL_SIZE > 0:
    pcd = pcd.voxel_down_sample(VOXEL_SIZE)
pcd.paint_uniform_color([0.55, 0.55, 0.55])

bbox = pcd.get_axis_aligned_bounding_box()
marker_radius = max(np.linalg.norm(bbox.get_extent()) * 0.01, 1e-4)

geometries = [pcd]
segment_colors = [
    [1.0, 0.15, 0.12],
    [0.1, 0.35, 1.0],
    [0.1, 0.75, 0.25],
    [1.0, 0.55, 0.05],
    [0.55, 0.2, 0.95],
    [0.0, 0.75, 0.8],
]

for idx, segment_points in enumerate(result.get("segment_point_clouds", [])):
    segment_pcd = o3d.geometry.PointCloud()
    segment_pcd.points = o3d.utility.Vector3dVector(np.array(segment_points, dtype=float))
    segment_pcd.paint_uniform_color(segment_colors[idx % len(segment_colors)])
    geometries.append(segment_pcd)

unassigned_points = np.array(result.get("unassigned_fit_points", []), dtype=float)
if len(unassigned_points) > 0:
    unassigned_pcd = o3d.geometry.PointCloud()
    unassigned_pcd.points = o3d.utility.Vector3dVector(unassigned_points)
    unassigned_pcd.paint_uniform_color([0.05, 0.05, 0.05])
    geometries.append(unassigned_pcd)

for idx, seg in enumerate(result["segments"]):
    cylinder_wire = make_cylinder_wireframe(seg, segment_colors[idx % len(segment_colors)])
    if cylinder_wire is not None:
        geometries.append(cylinder_wire)
    endpoints = np.array(seg["endpoints"], dtype=float)
    geometries.append(make_line(endpoints, segment_colors[idx % len(segment_colors)]))

for profile in result["terminal_end_profiles"]:
    center = np.array(profile["center"], dtype=float)
    circle = np.array(profile["profile_points"], dtype=float)
    circle_closed = np.vstack([circle, circle[0]])
    geometries.append(make_sphere(center, marker_radius, [1.0, 0.85, 0.05]))
    geometries.append(make_line(circle_closed, [0.1, 0.85, 0.25]))

for pair in result["joint_endpoint_pairs"]:
    for endpoint in pair["endpoints"]:
        center = np.array(endpoint["center"], dtype=float)
        geometries.append(make_sphere(center, marker_radius * 0.75, [0.95, 0.1, 0.9]))

o3d.visualization.draw_geometries(geometries)

## 5. 끝단 프로파일만 따로 꺼내기

포지셔너 고정 자세 계산 쪽으로 넘길 때는 아래 값들이 핵심입니다.

In [ ]:
end_a, end_b = result["terminal_end_profiles"]

for name, end in zip(["A", "B"], [end_a, end_b]):
    print(f"END {name}")
    print("  center:", np.array(end["center"]))
    print("  axis:  ", np.array(end["axis"]))
    print("  radius:", end["radius"])
    print("  profile_points shape:", np.array(end["profile_points"]).shape)
